# Wymagana konfiguracja w Azure Synapse serverless

Przed pierwszym uruchomieniem utwórz w Synapse serverless SQL następujące obiekty (dostosuj nazwy do wartości z komórki konfiguracyjnej):

```sql
CREATE DATABASE synapsedb_kjeden;
GO

USE synapsedb_kjeden;
GO

CREATE SCHEMA kaggle_aw;
GO

-- Credential do Azure Data Lake Storage Gen2
CREATE DATABASE SCOPED CREDENTIAL [adls_cred]
WITH
    IDENTITY = 'SHARED ACCESS SIGNATURE',
    SECRET = '<sv=...&spr=https&sig=...>';
GO

-- Źródło danych wskazujące na kontener gold w ADLS
CREATE EXTERNAL DATA SOURCE [gold]
WITH (
    LOCATION = 'https://<storage>.dfs.core.windows.net/gold',
    CREDENTIAL = [adls_cred]
);
GO

-- Format pliku parquet
CREATE EXTERNAL FILE FORMAT [parquet_file_format]
WITH (FORMAT_TYPE = PARQUET);
GO
```

In [0]:
silver_dir = dbutils.widgets.get("p_folder_silver")
gold_dir = dbutils.widgets.get("p_folder_gold")

'''silver_dir = "abfss://silver@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/"
gold_dir = "abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/"'''

silver_dir = silver_dir.rstrip("/") + "/"
gold_dir = gold_dir.rstrip("/") + "/"

gold_folder = gold_dir.rstrip("/").split("/")[-1] + "/"


_1
abfss://silver@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/
abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/
adventureworkskaggle/


In [0]:
import pyodbc

# -------------------------------------------------
# KONFIGURACJA: połączenie z Azure Synapse serverless
# -------------------------------------------------

key_vault_name = "kv-kacprusjeden-swec-001"

# --- UZUPEŁNIJ PONIŻSZE WARTOŚCI ---
synapse_server   = dbutils.secrets.get(key_vault_name, "synapse-serverless-srvname")
synapse_db       = dbutils.secrets.get(key_vault_name, "synapse-synapsedbkjeden")
synapse_schema   = "kaggle_aw"

# Nazwy obiektów utworzonych w Synapse serverless
synapse_data_source = "gold"
synapse_file_format = "delta_file_format"

# Poświadczenia SQL logowania do Synapse.
synapse_user     = dbutils.secrets.get(key_vault_name, "synapse-kjedentech-username")
synapse_password = dbutils.secrets.get(key_vault_name, "synapse-kjedentech-password")  # Verify the secret value is correct and accessible

synapse_conn_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER=tcp:{synapse_server},1433;"
    f"DATABASE={synapse_db};"
    f"UID={synapse_user};"
    f"PWD={synapse_password};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

try:
    pyodbc.connect(synapse_conn_string)
except Exception as e:
    print(f"Connection failed: {e}")
    raise

In [0]:
from pyspark.sql.types import *

def spark_type_to_sql(t):
    """Mapuje typ Spark na typ SQL dla Synapse serverless."""
    if isinstance(t, ByteType):      return "tinyint"
    if isinstance(t, ShortType):     return "smallint"
    if isinstance(t, IntegerType):   return "int"
    if isinstance(t, LongType):      return "bigint"
    if isinstance(t, FloatType):     return "real"
    if isinstance(t, DoubleType):    return "float"
    if isinstance(t, DecimalType):   return f"decimal({t.precision},{t.scale})"
    if isinstance(t, StringType):    return "varchar(max)"
    if isinstance(t, BooleanType):   return "bit"
    if isinstance(t, DateType):      return "date"
    if isinstance(t, TimestampType): return "datetime2"
    if isinstance(t, BinaryType):    return "varbinary(max)"
    return "varchar(max)"


def normalize_relative_location(location):
    return location.strip("/") + "/"


def build_external_table_ddl(table_name, location):
    """
    Buduje DDL CREATE EXTERNAL TABLE dla plików parquet zapisanych w kontenerze gold.
    Schemat kolumn jest generowany na podstawie DataFrame przekazanego do write_to_synapse.
    """
    # schemat kolumn będzie wstawiony przez write_to_synapse
    ddl_template = f"""
    IF OBJECT_ID('{synapse_schema}.{table_name}') IS NOT NULL
        DROP EXTERNAL TABLE {synapse_schema}.{table_name};

    CREATE EXTERNAL TABLE {synapse_schema}.{table_name} (
        {{columns}}
    )
    WITH (
        LOCATION = '{location}',
        DATA_SOURCE = {synapse_data_source},
        FILE_FORMAT = {synapse_file_format}
    );
    """
    return ddl_template


def execute_synapse_sql(sql):
    conn = pyodbc.connect(synapse_conn_string)
    cursor = conn.cursor()
    try:
        cursor.execute(sql)
        conn.commit()
    finally:
        cursor.close()
        conn.close()
    

def write_to_synapse(df, table_name, location, partition_by=None):
    relative_location = normalize_relative_location(location)
    target_path = f"{gold_dir}{relative_location.rstrip('/')}"

    # 1) Zapis danych do Azure Data Lake Storage jako parquet w kontenerze gold
    writer = (
        df.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
    )
    if partition_by:
        writer = writer.partitionBy(partition_by)
    writer.save(target_path)

    # 2) Generowanie schematu kolumn na podstawie DataFrame
    cols = ",\n        ".join([
        f"[{f.name}] {spark_type_to_sql(f.dataType)}"
        for f in df.schema.fields
    ])

    # 3) Utworzenie / zamiana external table w schemacie kaggle_aw
    ddl = build_external_table_ddl(table_name, relative_location).format(columns=cols)
    execute_synapse_sql(ddl)
    print(f"Zapisano parquet i zarejestrowano tabelę: {synapse_schema}.{table_name} -> {target_path}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import DateType

def read(name):
    return spark.read.format("delta").load(silver_dir + name)

product    = read("Product.csv")
reseller   = read("Reseller.csv")
territory  = read("Region.csv")
sales      = read("Sales.csv")
employee   = read("Salesperson.csv")
emp_region = read("SalespersonRegion.csv")
targets    = read("Targets.csv")

In [0]:
dim_product = product.select(
    "ProductKey",
    "Product",
    "Size",
    "Color",
    "Subcategory",
    "Category"
)

dim_reseller = reseller.select(
    "ResellerKey",
    "Reseller",
    "BusinessType",
    "City",
    "StateProvince",
    "CountryRegion",
    "CountryCode"
)

dim_territory = territory.select(
    "SalesTerritoryKey",
    "Region",
    "Country",
    "Group",
    "CountryCode"
)

dim_employee = employee.select(
    "EmployeeKey",
    "EmployeeID",
    "Salesperson",
    "Title",
    "UPN"
)

bridge_emp_terr = emp_region


for name, df in [
    ("DimProduct", dim_product),
    ("DimReseller", dim_reseller),
    ("DimSalesTerritory", dim_territory),
    ("DimEmployee", dim_employee),
    ("BridgeEmployeeTerritory", bridge_emp_terr)
]:
    write_to_synapse(df, name, gold_folder + name)

Zapisano parquet i zarejestrowano tabelę: kaggle_aw.DimProduct -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/DimProduct
Zapisano parquet i zarejestrowano tabelę: kaggle_aw.DimReseller -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/DimReseller
Zapisano parquet i zarejestrowano tabelę: kaggle_aw.DimSalesTerritory -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/DimSalesTerritory
Zapisano parquet i zarejestrowano tabelę: kaggle_aw.DimEmployee -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/DimEmployee
Zapisano parquet i zarejestrowano tabelę: kaggle_aw.BridgeEmployeeTerritory -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/BridgeEmployeeTerritory


In [0]:
date_from = sales.select(min("OrderDate")).first()[0]
date_to   = targets.select(max("TargetMonth")).first()[0]

dim_date = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{date_from}'),
                to_date('{date_to}'),
                interval 1 day
            )
        ) AS Date
        """
    )
    .select(
        (
            year("Date") * 10000
            + month("Date") * 100
            + dayofmonth("Date")
        ).cast("int").alias("DateSK"),

        "Date",

        year("Date").alias("Year"),
        quarter("Date").alias("Quarter"),
        month("Date").alias("Month"),
        date_format("Date", "MMMM").alias("MonthName"),
        dayofmonth("Date").alias("Day"),
        weekofyear("Date").alias("Week"),
        date_format("Date", "EEEE").alias("WeekdayName"),

        ((dayofweek("Date") + 5) % 7 + 1).alias("IsoDayOfWeek")
    )
)

write_to_synapse(dim_date, "DimDate", gold_folder + "DimDate")

Zapisano parquet i zarejestrowano tabelę: kaggle_aw.DimDate -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/DimDate


In [0]:
# --- KOMÓRKA 4: FactSales ---

fact_sales = (
    sales
    .withColumn(
        "OrderDateSK",
        (
            year("OrderDate") * 10000
            + month("OrderDate") * 100
            + dayofmonth("OrderDate")
        ).cast("int")
    )
    .withColumn(
        "OrderYearMonth",
        date_format("OrderDate", "yyyy-MM")
    )
    .withColumn(
        "Margin",
        col("Sales") - col("Cost")
    )
    .select(
        "SalesOrderNumber",
        "ProductKey",
        "ResellerKey",
        "EmployeeKey",
        "SalesTerritoryKey",
        "OrderDateSK",
        "OrderYearMonth",
        "Quantity",
        "UnitPrice",
        "Sales",
        "Cost",
        "Margin"
    )
    .dropDuplicates([
        "SalesOrderNumber",
        "ProductKey"
    ])
)

write_to_synapse(fact_sales, "FactSales", gold_folder + "FactSales", partition_by="OrderYearMonth")

Zapisano parquet i zarejestrowano tabelę: kaggle_aw.FactSales -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/FactSales


In [0]:
# --- KOMÓRKA 5: FactSalesTarget ---

fact_target = (
    targets
    .join(
        employee.select("EmployeeKey", "EmployeeID"),
        "EmployeeID",
        "left"
    )
    .withColumn(
        "TargetMonthSK",
        (
            year("TargetMonth") * 10000
            + month("TargetMonth") * 100
            + 1
        ).cast("int")
    )
    .select(
        "EmployeeKey",
        "TargetMonthSK",
        col("Target").alias("TargetAmount")
    )
    .dropDuplicates([
        "EmployeeKey",
        "TargetMonthSK"
    ])
)

write_to_synapse(fact_target, "FactSalesTarget", gold_folder + "FactSalesTarget")

Zapisano parquet i zarejestrowano tabelę: kaggle_aw.FactSalesTarget -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/FactSalesTarget


In [0]:
# --- KOMÓRKA 6: Marty ---

fs = spark.read.format("delta").load(gold_dir + "FactSales")
ft = spark.read.format("delta").load(gold_dir + "FactSalesTarget")
dt = spark.read.format("delta").load(gold_dir + "DimDate")
dp = spark.read.format("delta").load(gold_dir + "DimProduct")


# ============================================================
# Mart 1: wynik sprzedawcy miesiąc do miesiąca z realizacją celu
# ============================================================

sales_month = (
    fs
    .join(
        dt,
        fs.OrderDateSK == dt.DateSK
    )
    .groupBy(
        "EmployeeKey",
        "Year",
        "Month"
    )
    .agg(
        sum("Sales").alias("SalesAmount"),
        sum("Margin").alias("MarginAmount"),
        countDistinct("SalesOrderNumber").alias("OrderCount")
    )
)


target_month = (
    ft
    .join(
        dt,
        ft.TargetMonthSK == dt.DateSK
    )
    .select(
        "EmployeeKey",
        "Year",
        "Month",
        "TargetAmount"
    )
)


mart_perf = (
    sales_month
    .join(
        target_month,
        ["EmployeeKey", "Year", "Month"],
        "full"
    )
    .withColumn(
        "AttainmentPct",
        when(
            col("TargetAmount").isNotNull()
            & (col("TargetAmount") != 0),
            round(
                col("SalesAmount")
                / col("TargetAmount")
                * 100,
                2
            )
        )
        .otherwise(lit(None))
    )
    .orderBy(
        "Year",
        "Month",
        "EmployeeKey"
    )
)


write_to_synapse(mart_perf, "AggSalesEmployeeMonth", gold_folder + "AggSalesEmployeeMonth")


# ============================================================
# Mart 2: kategoria × region
# ============================================================

mart_prod_region = (
    fs
    .join(
        dp,
        "ProductKey"
    )
    .groupBy(
        "Category",
        "SalesTerritoryKey"
    )
    .agg(
        sum("Sales").alias("SalesAmount"),
        sum("Quantity").alias("UnitsSold")
    )
)


write_to_synapse(mart_prod_region, "AggSalesProductRegion", gold_folder + "AggSalesProductRegion")

Zapisano parquet i zarejestrowano tabelę: kaggle_aw.AggSalesEmployeeMonth -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/AggSalesEmployeeMonth
Zapisano parquet i zarejestrowano tabelę: kaggle_aw.AggSalesProductRegion -> abfss://gold@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/AggSalesProductRegion


In [0]:
# --- KOMÓRKA 7: kontrola jakości ---

checks = {
    "osierocone ProductKey":
        fs.join(
            dp,
            "ProductKey",
            "anti"
        ).count(),

    "osierocone EmployeeKey":
        fs.join(
            spark.read
                .format("delta")
                .load(gold_dir + "DimEmployee"),
            "EmployeeKey",
            "anti"
        ).count(),

    "ujemna marża (informacyjnie)":
        fs.filter(
            col("Margin") < 0
        ).count(),

    "Target bez sprzedawcy":
        ft.filter(
            col("EmployeeKey").isNull()
        ).count()
}


for k, v in checks.items():
    print(f"{k}: {v}")


assert (
    checks["osierocone ProductKey"] == 0
    and
    checks["osierocone EmployeeKey"] == 0
)

osierocone ProductKey: 0
osierocone EmployeeKey: 0
ujemna marża (informacyjnie): 20586
Target bez sprzedawcy: 0
